# Building MCP Clients: Capability Negotiation and Tool Discovery

An MCP client connects to one server and manages the lifecycle of that connection. The host application runs one client per server and coordinates tool calls across all connected clients.

## Client Lifecycle

MCP client connections follow a structured lifecycle:
1. **Connect**: establish transport (stdio pipe or HTTP/SSE connection)
2. **Initialize**: exchange capability declarations
3. **Discover**: fetch the server's tools, resources, and prompts
4. **Operate**: make calls and receive responses
5. **Disconnect**: clean shutdown

Capability negotiation during initialize determines what protocol features both sides support. A client that supports streaming declares it; a server that doesn't won't send streaming responses. This allows gradual protocol evolution without breaking compatibility.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import json

@dataclass
class MCPCapabilities:
    supports_streaming: bool = False
    supports_sampling: bool = False
    max_tool_calls_per_request: int = 10

@dataclass
class MCPConnection:
    server_name: str
    server_version: str
    tools: list = field(default_factory=list)
    resources: list = field(default_factory=list)
    prompts: list = field(default_factory=list)
    capabilities: MCPCapabilities = field(default_factory=MCPCapabilities)
    connected: bool = False

class MCPClient:
    def __init__(self, transport_fn: Callable, client_name: str = 'mcp-client', version: str = '1.0.0'):
        self.transport = transport_fn  # fn(method, params) -> dict
        self.name = client_name
        self.version = version
        self.conn: Optional[MCPConnection] = None

    def connect(self) -> MCPConnection:
        init_result = self.transport('initialize', {
            'protocolVersion': '2024-11-05',
            'clientInfo': {'name': self.name, 'version': self.version},
            'capabilities': {'sampling': {}}
        })
        if 'error' in init_result:
            raise ConnectionError(f'Init failed: {init_result["error"]}')
        info = init_result.get('serverInfo', {})
        self.conn = MCPConnection(
            server_name=info.get('name', 'unknown'),
            server_version=info.get('version', '0'),
        )
        self._discover()
        self.conn.connected = True
        return self.conn

    def _discover(self):
        tools_result = self.transport('tools/list', {})
        self.conn.tools = tools_result.get('tools', [])
        resources_result = self.transport('resources/list', {})
        self.conn.resources = resources_result.get('resources', [])

    def call_tool(self, name: str, args: dict) -> dict:
        if not self.conn or not self.conn.connected:
            raise RuntimeError('Not connected')
        if name not in {t['name'] for t in self.conn.tools}:
            return {'error': f'Tool {name} not available on this server'}
        return self.transport('tools/call', {'name': name, 'arguments': args})

    def read_resource(self, uri: str) -> dict:
        if not self.conn or not self.conn.connected:
            raise RuntimeError('Not connected')
        return self.transport('resources/read', {'uri': uri})

class MCPHost:
    def __init__(self):
        self.clients: dict = {}

    def add_server(self, alias: str, transport_fn: Callable) -> MCPConnection:
        client = MCPClient(transport_fn, 'host-client')
        conn = client.connect()
        self.clients[alias] = client
        return conn

    def all_tools(self) -> list:
        all_tools = []
        for alias, client in self.clients.items():
            if client.conn:
                for t in client.conn.tools:
                    all_tools.append({**t, 'server': alias})
        return all_tools

    def call(self, server_alias: str, tool_name: str, args: dict) -> dict:
        if server_alias not in self.clients:
            return {'error': f'Unknown server: {server_alias}'}
        return self.clients[server_alias].call_tool(tool_name, args)

# Demo: host connecting to two servers
from dataclasses import dataclass

# Reuse server from previous notebook (simplified)
class SimpleMCPServer:
    def __init__(self, name, tools_list):
        self._name = name
        self._tools = tools_list
    def handle(self, method, params):
        if method == 'initialize':
            return {'serverInfo': {'name': self._name, 'version': '1.0'}, 'capabilities': {}}
        if method == 'tools/list':
            return {'tools': self._tools}
        if method == 'resources/list':
            return {'resources': []}
        if method == 'tools/call':
            return {'content': [{'type': 'text', 'text': f'Result from {self._name}'}]}

math_server = SimpleMCPServer('math-server', [
    {'name': 'add', 'description': 'Add two numbers', 'inputSchema': {}},
    {'name': 'multiply', 'description': 'Multiply two numbers', 'inputSchema': {}},
])
files_server = SimpleMCPServer('files-server', [
    {'name': 'read_file', 'description': 'Read a file', 'inputSchema': {}},
])

host = MCPHost()
host.add_server('math', math_server.handle)
host.add_server('files', files_server.handle)

all_tools = host.all_tools()
print('All available tools across servers:')
for t in all_tools:
    print(f'  [{t["server"]}] {t["name"]}: {t["description"]}')

result = host.call('math', 'add', {'a': 5, 'b': 3})
print(f'\nTool call result: {result}')

## Dynamic Tool Discovery

A key MCP feature: tools can be registered and unregistered dynamically. A server can notify clients of capability changes via `notifications/tools/list_changed`. This enables:
- Contextual tools that appear only when relevant (a git tool that appears only when in a git repo)
- Per-user tool customization
- Progressive capability exposure based on authenticated permissions

Clients should re-discover tools after receiving change notifications rather than caching the initial list indefinitely.